In [ ]:
import sisl
from Zandpack.TimedependentTransport import TD_Transport
import numpy as np
import matplotlib.pyplot as plt
from Zandpack.FittingTools import find_correction
from ase.build import molecule

## Nanotube with H2O on the side
In this tutorial we use tight-binding to describe a carbon-nanotube while we simultaneously couple a water molecule onto the nanotube. The H2O molecules Hamiltonian has been calculated with a siesta calculation with a large cell. It gives an idea of the flexibility of the sisl objects and how to exploit this.

In [ ]:
_H_h2o = sisl.get_sile('siesta.TSHS').read_hamiltonian()
H_h2o  = _H_h2o.Hk()
g_h2o  = sisl.get_sile('siesta.TSHS').read_geometry()

a  = 1.44
tz = 5

C      =   sisl.Atom('C', R= 1.6)
_cnt   =   sisl.geom.nanotube(a,atoms = C, chirality = (8,0))
C      =  _cnt.tile(tz,2).center()
g_h2o  =   g_h2o.move(C).move([10,0,0])
_cnt2  =  _cnt.tile(tz,2)
cnt    =  _cnt2.add(g_h2o)


def func(self, ia, atoms, atoms_xyz=None):
    is_cnt = (np.linalg.norm(self.geometry.xyz[ia] - _cnt2.xyz,axis=1)<0.1).any()
    if is_cnt:
        idx = self.geometry.close(ia, R=[0.1, 1.44*1.1], atoms=atoms, atoms_xyz=atoms_xyz)
        self[ia, idx[0]] = 0
        self[ia, idx[1]] = -2.7
    else:
        pass

dev = sisl.Hamiltonian(cnt)
dev.set_nsc((1,1,1))
em  = sisl.Hamiltonian(_cnt.copy())
ep  = sisl.Hamiltonian(_cnt.copy().move(dev.cell[2,:] - em.cell[2,:]))
dev.construct(func)
em.construct( [[0.1, a * 1.1],[0.0, -2.7]])
ep.construct( [[0.1, a * 1.1],[0.0, -2.7]])

Hd = dev.Hk().toarray()
print(np.allclose(Hd, Hd.T.conj()))
offset = _cnt2.na
for io in range(H_h2o.shape[0]):
    for jo in range(H_h2o.shape[0]):
        dev[offset + io, offset+jo] = H_h2o[io,jo]
            
diff = np.linalg.norm(_cnt2.xyz - C, axis=1)
idx  = np.where(diff == diff.min())[0][0]
dev[offset+1,idx]  = 3.0
dev[idx, offset+1] = 3.0


eta  = 1j  * 1e-2
#line = np.linspace(-8, 8, 250) + eta  + 1e-5
line = np.hstack((np.arange(-8,-1,0.25), np.arange(-1,1, 0.05),np.arange(1,8, 0.25)))
#print(line.shape)

line = np.vstack((line,line))+eta+1e-4

    
    
D = TD_Transport([em.geometry,ep.geometry], dev, kT_i = [0.025, 0.025])
D.Make_Contour(line, 20, pole_mode = 'JieHu2011')
plt.show()

D.Electrodes( semi_infs = ['-a3', '+a3'] , kp = [[1,1,50], [1,1,50]])
D.make_device()
D.run_electrodes(fois_gras_H = [em,ep])
dev_rear = dev.sub(D.Device._rearange_indices)
D.run_device(    fois_gras_H = dev_rear)
xyz_N = np.array([dev_rear.xyz[dev_rear.o2a(i)] for i in range(dev_rear.no)])
np.save('Pos_Prepivot.npy',xyz_N, )

D.Device.Visualise(axes=[0,2])
D.read_data()
D.Device.Visualise(axes = [0,2])
plt.show()
plt.plot(line[0],D.tbtT[D.sampling_idx[0]])

In [ ]:
#D.reset_all_fits() # Comment in to reset fits!
# In[]
Eg = np.linspace(-8,8,300)
#Test.reset_all_fits()
Emin,Emax = -7,7

NL = 41
E  = np.linspace(Emin, Emax,NL)

E[0:NL//2] -= .5
E[NL//2+1:]+= .5


E1 = E.copy()
E2 = E.copy()


init_E = [E1[None,:],E2[None,:]]
number = 0.2*(41/NL)**0.5
G = np.ones(init_E[0].shape)*number
G[0,(NL//2-3):(NL//2+4)] = number/2
G[0,NL//2] = number/30


G1 = G.copy()
G2 = G.copy()


init_G = [G1,G2]
alpha_PO = 0.001

min_tol = np.zeros((1,NL))

def run_mini(its,method, elecs = None):
    D.Fit(fact = 0.6,                            # Redundant when we give init_E and init_G
          Fallback_W = 5.0,                    # Redundant\n",
          NumL = NL,                           # Not redundant\n",
          fit_mode = 'all',                    # Important to choose mode\n",
          force_PSD     = True,                # the self-energies are not positive semidefinite
          force_PSD_tol = [min_tol,
                           min_tol], # 
          use_analytical_jac = False,          # Important for speed
          min_method = method,                 # Choose from any scipy.optimize.minimize method
          ebounds = (-7.5,7.5),                # bounds on centres
          wbounds = (0.0001, 3),               # bounds on widths
          gbounds = (None, None),              # bounds on sizes, redundant right now
          tol = 1e-5,                          # any negative value with mean we fit all matrix elements of \\Gamma
          options = {'disp':True,              # Minimizer options
                     'maxiter':its, 
                     'gtol':1e-10, 
                     #'iprint':1,
                     },
          
          fit_real_part = False,               # 
          alpha_PO      = alpha_PO,            # Repulsion
          init_E        = init_E,              # Give initial ei's and gi's
          init_G        = init_G,
          which_e       = elecs
          )
    C = find_correction(D)
    D.Renormalise_H(C)
print(E1)

In [ ]:
#D.reset_all_fits()
run_mini(0,'nelder-mead')
print('Blocks of first electrode')
print(D.fitted_lorentzians[0].is_zero)
print('Blocks of second electrode')
print(D.fitted_lorentzians[1].is_zero)

# We access the blocks of the fitted matrices like this:
# only block (0,0) is nonzero
for iL in [17,18,19,21,22,23]:
    for i,v in enumerate(D.fitted_lorentzians[0].vals):
        v[:,iL]*=0.5
        print(D.fitted_lorentzians[0].inds[i])
    for v in D.fitted_lorentzians[1].vals:
        v[:,iL]*=0.5
        print(D.fitted_lorentzians[1].inds[i])


In [ ]:
IK = 0
D.Inspect_Lorentzian_fit(0,0,0,0,2, ik = IK, Emin = -9, Emax = 9, center_lines = False, n_samples=1000)
plt.xlabel(r'$E-E_F$ [eV]')
plt.ylim([-5,5])
#plt.xlim([-1,1])
plt.show()
D.Inspect_SE_from_hilbert_transform(0,0,0,2,2, ik = IK, Emin = -9, Emax = 9, n_samples=1000)
plt.xlabel(r'$E-E_F$ [eV]')
plt.ylim([-50,50])
#plt.xlim([-1,1])
plt.show()

#D.Inspect_Lorentzian_fit(1,11,11,1,0,                          ik = IK, Emin = -7, Emax = 7)
#plt.show()
#D.Inspect_Lorentzian_fit(0,0,0,1,1,                            ik = IK, Emin = -7, Emax = 7)
#plt.show()
#D.Inspect_Lorentzian_fit(0,0,0,1,0,                            ik = IK, Emin = -7, Emax = 7)


In [ ]:
D.tofile('NR')
D.Inspect_transmission_from_hilbert_transform(E=Eg,eta=1e-3, NO=False)
plt.xlim([-5,5])
#D.Check_input_to_ODE()

In [ ]:
D.tofile('NT')

In [ ]:
!SCF Dir=$PWD file=NT
!psinought Dir=$PWD file=NT add_random=True random_weight=0.00001 steptol=1e-5


In [ ]:
!mpirun -np 3 SuperZand Dir=$PWD